# 전체 분석 Pipeline

## 1. 데이터 전처리

원본 데이터에서 결측치 및 이상치를 확인하고 분석에 사용할 변수를 정리하였다. 이후 Target 변수의 분포를 확인한 결과, 전체 데이터에서 불만족 고객의 비율이 약 4% 수준으로 나타나 심각한 클래스 불균형 문제가 존재함을 확인하였다.

## 2. 후보 변수 선정

초기 313개의 변수를 대상으로 통계적 방법과 머신러닝 기반 방법을 함께 적용하였다.

* 통계적 변수 선택
* 상관관계 기반 중복 변수 제거
* LASSO 변수 선택
* Random Forest 변수 중요도 기반 선택
* XGBoost 변수 중요도 기반 선택

서로 다른 변수 선택 방법에서 반복적으로 선택되는 변수를 확인하여 모델에 중요한 변수들을 선별하였다.

## 3. 변수 중복 제거 및 최종 Feature 구성

상관관계가 높은 변수들을 그룹화하고 각 그룹에서 대표 변수를 선정하였다.

이후 통계적 방법과 머신러닝 방법의 결과를 종합하여 최종 Feature Set을 구성하였다.

최종적으로 **313개의 초기 변수에서 15개의 핵심 변수**를 선정하였다.

또한 최종 15개 변수 중 **13개가 통계검정, LASSO, Random Forest, XGBoost의 4개 방법에서 공통적으로 선택된 변수**로 확인되어 변수 선정의 일관성을 확보하였다.

## 4. Train / Validation / Test 분할

데이터를 다음과 같이 분할하였다.

* Train: 45,612건
* Validation: 15,204건
* Test: 15,204건

Validation 데이터와 Test 데이터는 모델 학습 과정에서 사용하지 않고 성능 평가를 위해 별도로 유지하였다.

## 5. 클래스 불균형 처리

Train 데이터의 Target 분포는 다음과 같았다.

* 정상 고객: 43,807명
* 불만족 고객: 1,805명
* 불만족 비율: 약 3.96%

불만족 고객의 비율이 매우 낮기 때문에 Train 데이터에만 SMOTE를 적용하였다.

SMOTE 적용 후:

* 정상 고객: 43,807명
* 불만족 고객: 43,807명
* 총 학습 데이터: 87,614건

Validation과 Test 데이터에는 SMOTE를 적용하지 않아 실제 데이터 분포에서 모델을 평가하였다.

## 6. Logistic Regression Baseline

Logistic Regression을 baseline 모델로 설정하였다.

스케일링과 SMOTE를 적용한 Logistic Regression을 통해 선형 모델의 성능을 확인하고 이후 XGBoost와 비교하였다.

Validation 기준 Logistic Regression의 성능은 다음과 같다.

* ROC-AUC: 0.7984
* PR-AUC: 0.1340
* Precision: 0.0870
* Recall: 0.7508
* F1-score: 0.1559

Logistic Regression은 높은 Recall을 보였지만 Precision과 F1-score가 낮게 나타났다.

## 7. XGBoost 모델 구축

비선형적인 변수 관계와 변수 간 상호작용을 반영하기 위해 XGBoost를 적용하였다.

Baseline XGBoost는 Logistic Regression보다 우수한 성능을 나타냈다.

* ROC-AUC: 0.8172
* PR-AUC: 0.1615
* Precision: 0.1678
* Recall: 0.4908
* F1-score: 0.2501

특히 ROC-AUC와 PR-AUC가 Logistic Regression보다 개선되어 XGBoost를 최종 후보 모델로 선정하였다.

## 8. Leakage-Free Hyperparameter Tuning

XGBoost의 성능을 추가적으로 개선하기 위해 Randomized Search 기반 하이퍼파라미터 탐색을 수행하였다.

최적 파라미터는 다음과 같다.

* n_estimators: 300
* max_depth: 4
* learning_rate: 0.01
* min_child_weight: 5
* subsample: 1.0
* colsample_bytree: 0.6
* gamma: 0.01

Leakage-Free 방식의 Validation 성능은 다음과 같다.

* ROC-AUC: 0.8208
* PR-AUC: 0.1677

## 9. Threshold Optimization

기본 분류 기준인 0.5를 그대로 사용하지 않고 Validation 데이터에서 여러 threshold를 비교하였다.

F1-score를 기준으로 최적 threshold를 탐색한 결과 **0.70**을 최종 threshold로 선정하였다.

Threshold 0.70에서 Validation 성능:

* Precision: 0.2063
* Recall: 0.4435
* F1-score: 0.2816

이를 통해 단순히 많은 고객을 불만족으로 분류하기보다 실제 불만족 가능성이 높은 고객을 선별하는 방향으로 분류 기준을 조정하였다.

## 10. 최종 Test 평가

최종적으로 Test 데이터에 단 한 번 적용하여 모델의 일반화 성능을 평가하였다.

### Final XGBoost Test Performance

* ROC-AUC: **0.8172**
* PR-AUC: **0.1496**
* Precision: **0.1877**
* Recall: **0.4326**
* F1-score: **0.2618**
* Accuracy: 0.9036

Confusion Matrix:

```text
[[13478  1125]
 [  341   260]]
```

실제 불만족 고객 601명 중 260명을 정확하게 식별하였다.

## 11. SHAP 기반 모델 해석

최종 XGBoost 모델의 예측 근거를 해석하기 위해 SHAP 분석을 수행하였다.

주요 변수는 다음과 같다.

1. var15
2. ind_var30
3. num_meses_var5_ult3
4. saldo_medio_var5_hace3
5. saldo_var30
6. num_var35
7. var38
8. num_var22_ult1
9. saldo_var42
10. saldo_medio_var5_ult1

SHAP 분석을 통해 단순히 어떤 모델이 높은 성능을 내는지를 확인하는 것에서 나아가, **어떤 변수들이 고객 불만족 예측에 영향을 미치는지**를 확인하였다.

## 12. 최종 분석 구조

전체 분석 과정은 다음과 같이 요약할 수 있다.

**313개 초기 변수**

↓

**통계검정 + 상관관계 분석 + LASSO + Random Forest + XGBoost**

↓

**중복 변수 제거 및 대표 변수 선정**

↓

**최종 15개 Feature**

↓

**Train / Validation / Test 분리**

↓

**Train 데이터에만 SMOTE 적용**

↓

**Logistic Regression Baseline**

↓

**XGBoost**

↓

**Leakage-Free Hyperparameter Tuning**

↓

**Threshold Optimization**

↓

**최종 XGBoost Test 평가**

↓

**SHAP 기반 모델 해석**

↓

**고객 불만족 가능성이 높은 고객 식별**

## 13. 최종 목적

본 분석의 최종 목적은 단순히 높은 Accuracy를 얻는 것이 아니라, 전체 고객 중 소수인 **불만족 가능성이 높은 고객을 사전에 식별하여 우선적인 관리 대상으로 선별하는 것**이다.

따라서 최종 모델은 ROC-AUC와 PR-AUC를 통한 판별 성능뿐만 아니라 Precision, Recall, F1-score 및 SHAP 기반 해석 결과를 함께 고려하여 평가하였다.
